In [1]:
import pandas as pd

In [2]:
df = pd.read_json('1-1.여성의류(196).json')
df.head(5)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."
2,1025044,오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다. 키 158...,SNS,패션,여성의류,OO 아** 구스코트,314,78,1,"[{'Aspect': '사이즈', 'SentimentText': '키 158센티로에..."
3,1025046,이웃님들 오늘도 안녕하신가요? 오늘은 따끈한 신상 패딩 후기 올려봅니다~~ 겨울이...,SNS,패션,여성의류,OO 아** 구스코트,307,74,1,"[{'Aspect': '색상', 'SentimentText': '흰색 패딩이 너무나..."
4,1025071,OOO 구스로 소문난 OO의 롱패딩~ 한번 구경 가봐요. 일단 보는 순간 고급스럽...,SNS,패션,여성의류,OO 아** 구스코트,279,68,1,"[{'Aspect': '소재', 'SentimentText': ' 일단 보는 순간 ..."


### 문제
- 데이터프레임에서 'Aspects' 컬럼에 데이터들을 이용하여 분류 모델 생성
- SentimentText 텍스트를 이용하여 'Aspect', 'SentimentPolarity'의 값들을 예측
    1. df에서 'Aspects' 데이터를 추출
    2. SentimentText 데이터를 문자형으로 이루어져있으니 학습에 대한 데이터의 형태로 변환 (문자의 데이터를 숫자형 데이터) -> 토큰화(Okt), 벡터화(TF-IDF)
    3. 종속 변수는 'Aspect', 'SentimentPolarity'
    4. 분류 모델(LinearSVC) randomstate = 42 고정
    5. 테스트 데이터를 이용하여 분류가 잘되고 있는가? 정확도(acc)확인

    벡터화, 모델링 파이프라인으로 연결해서 사용

In [3]:
test_data = [
    '색상이 마음에 든다',
    '설명에 비해 옷이 두껍진 않다',
    '길이가 너무 길지도 않고 짧지도 않다'
]

In [4]:
df['Aspects']

0      [{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인...
1      [{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사...
2      [{'Aspect': '사이즈', 'SentimentText': '키 158센티로에...
3      [{'Aspect': '색상', 'SentimentText': '흰색 패딩이 너무나...
4      [{'Aspect': '소재', 'SentimentText': ' 일단 보는 순간 ...
                             ...                        
118    [{'Aspect': '두께', 'SentimentText': '저는 두께가 얇은 ...
119    [{'Aspect': '기능', 'SentimentText': '겨울에 따뜻하게 입...
120    [{'Aspect': '소재', 'SentimentText': '이 목폴라 티셔츠는...
121    [{'Aspect': '소재', 'SentimentText': '일단 원단이 너무 ...
122    [{'Aspect': '활용성', 'SentimentText': '캐쥬얼 코디를 데...
Name: Aspects, Length: 123, dtype: object

In [5]:
test_df = df.explode('Aspects').copy()

In [6]:
test_df.head(3)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인데..."
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"{'Aspect': '두께', 'SentimentText': '이것만 입기엔 얇지만..."
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"{'Aspect': '기능', 'SentimentText': '초겨울까지는 운동 갈..."


In [7]:
new_cols = test_df['Aspects'].apply(pd.Series)
# Aspects 컬럼을 분리하여 새로운 데이터프레임 생성
test_df = test_df.drop('Aspects', axis=1).join(new_cols)
# 기존 Aspects 컬럼 삭제 후 분리된 컬럼 합치기

In [8]:
test_df.head(3)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,두께,이것만 입기엔 얇지만,3,-1
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1


In [9]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [10]:
df_model = test_df[['Aspect', 'SentimentPolarity']].copy()
df_model

,Aspect,SentimentPolarity
0,디자인,1
0,두께,-1
0,기능,1
0,색상,0
0,디자인,0
...,...,...
122,길이,1
122,활용성,1
122,디자인,1
122,품질,1


In [11]:
y1 = df_model['Aspect']
le = LabelEncoder()
y1 = le.fit_transform(y1)
# 텍스트 데이터를 숫자 레이블로 변환
df_y1 = pd.DataFrame(y1)

y2 = df_model['SentimentPolarity']
df_y2 = pd.DataFrame(y2)

In [12]:
sentimenttext_data = test_df['SentimentText']
# sentimenttext_data --> Series 데이터를 학습 데이터로 설정하면 데이터 누수 발생?? 정확도 1나옴
df_data_sentimenttext =  pd.DataFrame(sentimenttext_data)
df_data_sentimenttext

,SentimentText
0,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서
0,이것만 입기엔 얇지만
0,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.
0,색상도 디자인도 무난해서
0,디자인도 무난해서
...,...
122,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.
122,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.
122,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고
122,퀄리티도 넘 휼륭합니다.


In [13]:
# 토큰화 -> 벡터화
# 토큰화
okt = Okt()

def tokenize(text):
    return okt.morphs(text)

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=tokenize,
    ngram_range=(1,2)
)
# 벡터화
vectorizer = TfidfVectorizer(lowercase=False, tokenizer=tokenize, ngram_range=(1,2))


In [14]:
X_vec = vectorizer.fit_transform(test_df['SentimentText'])

X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X_vec, df_y1,df_y2, test_size=0.2, random_state=42
)

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
model_1 = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
model_2 = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
model_1.fit(X_train, y1_train.values.ravel())
model_2.fit(X_train, y2_train.values.ravel())

y1_pred = model_1.predict(X_test)
y2_pred = model_2.predict(X_test)

acc1 = accuracy_score(y1_test, y1_pred)
acc2 = accuracy_score(y2_test, y2_pred)
print(f'Aspect 분류 정확도: {acc1:.4f}')
print(f'SentimentPolarity 분류 정확도: {acc2:.4f}')

Aspect 분류 정확도: 1.0000
SentimentPolarity 분류 정확도: 1.0000


In [16]:
# 1. 예측할 새로운 문장 데이터
test_data = [
    '색상이 마음에 든다',
    '설명에 비해 옷이 두껍진 않다',
    '길이가 너무 길지도 않고 짧지도 않다'
]

# 2. (중요) Cell 80에서 학습한 'vectorizer'를 그대로 사용
#    .transform()을 사용하면 13,750개 특성 기준으로 변환됨
X_new_vec = vectorizer.transform(test_data) 

# 3. 모델로 예측
pred_y1_encoded = model_1.predict(X_new_vec)
pred_y2 = model_2.predict(X_new_vec)

# 4. le를 이용해 Aspect 예측(숫자)을 텍스트로 변환
pred_y1_text = le.inverse_transform(pred_y1_encoded)

# 5. 결과 출력
print("---새로운 문장 예측 결과 ---")
for i in range(len(test_data)):
    print(f"입력 문장: {test_data[i]}")
    print(f"  -> 예측 Aspect: {pred_y1_text[i]}")
    print(f"  -> 예측 Polarity: {pred_y2[i]}\n")

---새로운 문장 예측 결과 ---
입력 문장: 색상이 마음에 든다
  -> 예측 Aspect: 색상
  -> 예측 Polarity: 1

입력 문장: 설명에 비해 옷이 두껍진 않다
  -> 예측 Aspect: 디자인
  -> 예측 Polarity: 1

입력 문장: 길이가 너무 길지도 않고 짧지도 않다
  -> 예측 Aspect: 길이
  -> 예측 Polarity: 1

